# Generates topic summary

Printable table at: https://docs.google.com/spreadsheets/d/1LUjjiwLPjJQqMm5nD77W_HJ6IjPgyGMpNQzVNoJBg70/edit?gid=746607918#gid=746607918

Pritable table 2018: https://docs.google.com/spreadsheets/d/1TFck_XpLa_W86hx-pGUL1CS2CRCFhcquLip2oI_ONXs/edit?gid=2046672856#gid=2046672856

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
base_path = Path("data")
years = list(range(13, 23))
year_labels = [2000 + year for year in years]
topic_columns = ["Topic_1", "Topic_2", "Topic_3"]

# Keep only compact per-topic aggregates; never concatenate all articles or comments.
aggregate_frames = []
total_articles = 0
total_comments = 0

for year, year_label in zip(years, year_labels):
    articles_year = pd.read_csv(
        base_path / f"articles_file_{year:02d}.tsv",
        sep="\t",
        dtype={"Article_Id": "string", **{column: "string" for column in topic_columns}},
        keep_default_na=False,
        usecols=["Article_Id", *topic_columns],
    )

    comments_per_article = None
    for comment_path in sorted(base_path.glob(f"comments_file_*_{year:02d}.tsv")):
        file_counts = pd.Series(dtype="int64")
        for comments_chunk in pd.read_csv(
            comment_path,
            sep="\t",
            dtype={"Article_Id": "string"},
            usecols=["Article_Id"],
            chunksize=250_000,
        ):
            chunk_counts = comments_chunk.groupby("Article_Id", sort=False).size()
            file_counts = file_counts.add(chunk_counts, fill_value=0)
        comments_per_article = file_counts if comments_per_article is None else comments_per_article.add(file_counts, fill_value=0)

    if comments_per_article is None:
        comments_per_article = pd.Series(dtype="int64", name="num_comments")
    else:
        comments_per_article = comments_per_article.astype("int64").rename("num_comments")

    articles_year["num_comments"] = (
        articles_year["Article_Id"].map(comments_per_article).fillna(0).astype("int64")
    )
    total_articles += len(articles_year)
    total_comments += int(comments_per_article.sum())

    for level, topic_column in enumerate(topic_columns):
        group_columns = topic_columns[: level + 1]
        valid = articles_year.loc[articles_year[topic_column].ne(""), ["Article_Id", *group_columns, "num_comments"]]
        yearly_summary = (
            valid.groupby(group_columns, as_index=False, sort=False)
            .agg(
                num_articles=("Article_Id", "nunique"),
                num_comments=("num_comments", "sum"),
            )
            .rename(columns=dict(zip(group_columns, ["topic1", "topic2", "topic3"])))
        )
        for column in ["topic1", "topic2", "topic3"]:
            if column not in yearly_summary:
                yearly_summary[column] = ""
        yearly_summary["year"] = year_label
        aggregate_frames.append(yearly_summary[["topic1", "topic2", "topic3", "year", "num_articles", "num_comments"]])

    del articles_year, comments_per_article

print(f"Number of articles: {total_articles}")
print(f"Number of comments: {total_comments}")

Number of articles: 586942
Number of comments: 75644850


In [ ]:
# Combine yearly topic aggregates and add one article-count column per year.
yearly_topic_summary = pd.concat(aggregate_frames, ignore_index=True)
group_columns = ["topic1", "topic2", "topic3"]
topic_summary = yearly_topic_summary.groupby(group_columns, as_index=False, sort=False).agg(
    num_articles=("num_articles", "sum"),
    num_comments=("num_comments", "sum"),
)
article_year_counts = (
    yearly_topic_summary.pivot_table(index=group_columns, columns="year", values="num_articles", aggfunc="sum", fill_value=0)
    .rename(columns=lambda year: f"num_articles_{year}")
    .reset_index()
)
topic_summary = topic_summary.merge(article_year_counts, on=group_columns, how="left")
for year_label in year_labels:
    column = f"num_articles_{year_label}"
    if column not in topic_summary:
        topic_summary[column] = 0
    topic_summary[column] = topic_summary[column].fillna(0).astype("int64")

topic_summary["topic_level"] = topic_summary[group_columns].ne("").sum(axis=1)
topic1_order = topic_summary.loc[topic_summary["topic_level"].eq(1)].sort_values(["num_comments", "topic1"], ascending=[False, True])["topic1"].tolist()
topic1_rank = {topic: rank for rank, topic in enumerate(topic1_order)}
topic2_order = (topic_summary.loc[topic_summary["topic_level"].eq(2)].sort_values(["topic1", "num_comments", "topic2"], ascending=[True, False, True]).groupby("topic1", sort=False)["topic2"].apply(list).to_dict())
topic2_rank = {(parent, topic): rank for parent, topics in topic2_order.items() for rank, topic in enumerate(topics)}
topic3_order = (topic_summary.loc[topic_summary["topic_level"].eq(3)].sort_values(["topic1", "topic2", "num_comments", "topic3"], ascending=[True, True, False, True]).groupby(["topic1", "topic2"], sort=False)["topic3"].apply(list).to_dict())
topic3_rank = {(parent1, parent2, topic): rank for (parent1, parent2), topics in topic3_order.items() for rank, topic in enumerate(topics)}
topic_summary["topic1_rank"] = topic_summary["topic1"].map(topic1_rank)
topic_summary["topic2_rank"] = [-1 if level == 1 else topic2_rank[(topic1, topic2)] for level, topic1, topic2 in topic_summary[["topic_level", "topic1", "topic2"]].itertuples(index=False, name=None)]
topic_summary["topic3_rank"] = [-1 if level < 3 else topic3_rank[(topic1, topic2, topic3)] for level, topic1, topic2, topic3 in topic_summary[["topic_level", "topic1", "topic2", "topic3"]].itertuples(index=False, name=None)]
topic_summary = (topic_summary.sort_values(["topic1_rank", "topic2_rank", "topic_level", "topic3_rank"], ascending=True).drop(columns=["topic_level", "topic1_rank", "topic2_rank", "topic3_rank"]).reset_index(drop=True))

display(topic_summary)
print(f"Topic summary rows: {len(topic_summary)}")

,topic1,topic2,topic3,num_articles,num_comments,num_articles_2013,num_articles_2014,num_articles_2015,num_articles_2016,num_articles_2017,num_articles_2018,num_articles_2019,num_articles_2020,num_articles_2021,num_articles_2022
0,inland,,,39445,12548484,5186,4529,4401,4775,5043,3930,3739,2384,2488,2970
1,inland,parteien,,7607,3357763,934,784,635,829,1500,869,1099,355,295,307
2,inland,parteien,spoe,1506,682670,149,161,146,273,267,171,177,71,49,42
3,inland,parteien,fpoe,1429,642664,188,141,146,146,145,179,331,68,43,42
4,inland,parteien,oevp,1309,636156,120,192,117,135,245,95,159,40,77,129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2102,testbereich-presales,testpromo-bankaustria2,,1,0,0,0,0,0,0,1,0,0,0,0
2103,testbereich-presales,testpromo-erneuerbareenergien,,2,0,0,0,0,0,0,2,0,0,0,0
2104,testbereich-presales,testpromo-tu,,4,0,0,0,0,0,0,4,0,0,0,0
2105,testbereich-presales,testpromo-wwtf,,2,0,0,0,0,0,0,2,0,0,0,0


Topic summary rows: 2107


In [11]:
unique_topic_counts = {
    column: topic_summary.loc[topic_summary[column].ne(""), column].nunique()
    for column in ["topic1", "topic2", "topic3"]
}

print(unique_topic_counts)

{'topic1': 25, 'topic2': 304, 'topic3': 1721}


In [4]:
# Empty strings represent missing hierarchy levels.
topic_summary[group_columns] = (
    topic_summary[group_columns]
    .fillna("")
    .astype("string")
    .apply(lambda column: column.str.strip())
)

topic_depth = (topic_summary[group_columns].ne("").sum(axis=1))

# The final row is always a leaf. Every other row is a leaf when
# the following row does not go deeper.
next_topic_depth = topic_depth.shift(-1)

topic_summary["is_leaf"] = (
    next_topic_depth.le(topic_depth)
    | next_topic_depth.isna()
)

,topic1,topic2,topic3,num_articles,num_comments,num_articles_2013,num_articles_2014,num_articles_2015,num_articles_2016,num_articles_2017,num_articles_2018,num_articles_2019,num_articles_2020,num_articles_2021,num_articles_2022,is_leaf
0,inland,,,39445,12548484,5186,4529,4401,4775,5043,3930,3739,2384,2488,2970,False
1,inland,parteien,,7607,3357763,934,784,635,829,1500,869,1099,355,295,307,False
2,inland,parteien,spoe,1506,682670,149,161,146,273,267,171,177,71,49,42,True
3,inland,parteien,fpoe,1429,642664,188,141,146,146,145,179,331,68,43,42,True
4,inland,parteien,oevp,1309,636156,120,192,117,135,245,95,159,40,77,129,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2102,testbereich-presales,testpromo-bankaustria2,,1,0,0,0,0,0,0,1,0,0,0,0,True
2103,testbereich-presales,testpromo-erneuerbareenergien,,2,0,0,0,0,0,0,2,0,0,0,0,True
2104,testbereich-presales,testpromo-tu,,4,0,0,0,0,0,0,4,0,0,0,0,True
2105,testbereich-presales,testpromo-wwtf,,2,0,0,0,0,0,0,2,0,0,0,0,True


In [6]:
topic_summary.to_csv("my_data/topic_summary_2013_2022.csv", sep=",", index=False)